# BDB27 — Combine ceiling-binding pipeline (full run)

**Does the Combine ceiling bind on max-effort routes — and does incentive exposure shape the gap?**
(WR primary, TE replication)

This notebook is the *runnable* deliverable: it opens, you set two things in the
**Configuration** cell below (`MODE` and — only if needed — `DATA_DIR`), then **Run All**.
No terminal required.

It drives the `src/` engine end-to-end in the order

```
01_audit → 02_combine_envelope → 06_security → 00_thresholds → 03_game_demand →
07_security_models → 00b_external_proxies → 04_models → 05_validate
```

and renders the **≤5 writeup figures** into `outputs/figures/`.

**Works on** the ARM64 Raspberry Pi, a Windows / macOS laptop, and **Kaggle**
(auto-detects the attached dataset under `/kaggle/input/...`; otherwise uses `data/raw`).

### Reading the labels

* `sample` mode outputs are stamped **`UNVALIDATED SAMPLE OUTPUT`** — plumbing, not findings.
* `full` mode outputs are stamped **`FULL`** — the real analysis.
* synthetic-fixture outputs are stamped **`SYNTHETIC FIXTURE (not a finding)`** — they prove the
  plumbing, nothing about the NFL.

### D38 — the security spine

Analysis A (the old "did the market reward speed?" check) is **retired as a gate**; it is a short
descriptive appendix only. The paper's spine is **incentive exposure**: a player whose draft stock is
slipping entering the Combine has a stronger reason to invest in the speed mechanism, so he should
show a **smaller** Combine→game gap than a high/secure player. Groups are assigned from
**pre-Combine** EDP only (`moved_down` vs `high_secure`); post-Combine EDP is never an exposure.
A **positive** group coefficient = `moved_down` has the smaller gap (the hypothesis).

## 1. Configuration

Edit only this cell. `MODE="sample"` runs everything that the Pi's partial data allows and
clearly flags the game-side stages as `PENDING FULL RUN`. Set `MODE="full"` once the full
`game_tracking_{2023,24,25}.csv` files are available.

In [ ]:
# ===================== CONFIG — edit me (nothing else) =====================
# Game-side per-frame speed is despiked at game.speed_cap_yds (12.5 yd/s; frame-level data-QC despike, D35) — re-run the game chain (FORCE/CLEAN) after a QC change.
MODE               = "sample"   # "sample" | "full"
DATA_DIR           = None       # None = auto-detect (/kaggle/input/... else <repo>/data/raw)
FORCE              = False      # True = ignore stage checkpoints and rebuild
CLEAN              = True       # True = clear previous stage outputs for a reproducible run
RUN_SYNTHETIC_DEMO = True       # exercise the game-side path on the committed fixtures
SYNTHETIC_REGIMES  = ("binding", "null")            # ceiling fixture (04/05)
SECURITY_REGIMES   = ("planted", "null", "selection")  # security fixture (07)
SHOW_FIGURES       = True       # display figures inline
FIG_DPI            = 150
assert MODE in ("sample", "full"), MODE
# ==========================================================================

## 2. Bootstrap — locate repo, data, write the run config

No commands to type: this cell finds the repository, picks the data folder, writes a
scratch run-config (`bdb27_run/config.yaml`) that points the engine at the data, and prints
what will run.

In [ ]:
import importlib, io, json, sys, time, traceback, shutil
from contextlib import redirect_stdout, redirect_stderr
from pathlib import Path
import numpy as np
import pandas as pd
import yaml

# ---- locate the repo root (config.yaml + src/common.py) --------------------
def find_repo_root():
    seen = []
    c = Path.cwd().resolve()
    for p in [c, *c.parents]:
        if (p / "config.yaml").exists() and (p / "src" / "common.py").exists():
            return p
    raise RuntimeError("Cannot locate the BDB27 repo root (need config.yaml + src/common.py) "
                       "— open this notebook from inside the BDB27 checkout.")

REPO = find_repo_root()
SRC  = REPO / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))          # so `import common` etc. resolve (numeric stage names import via importlib)
import common                             # the shared engine

# bind the logger to the real stderr BEFORE any redirect, so stage logs stay visible
common.get_logger()
CFG_REPO = common.load_config(str(REPO / "config.yaml"))

# ---- pick the data directory ----------------------------------------------
def has_combine(d):
    return Path(d).is_dir() and (Path(d) / "combine_tracking.csv").exists()

def detect_data_dir(override):
    if override:
        p = Path(override).expanduser().resolve()
        if has_combine(p):
            return p
        hits = sorted(p.rglob("combine_tracking.csv"))
        return hits[0].parent if hits else p
    kaggle = Path("/kaggle/input")
    if kaggle.exists():
        hits = sorted(kaggle.rglob("combine_tracking.csv"))
        if hits:
            return hits[0].parent
    return REPO / "data" / "raw"

DATA = detect_data_dir(DATA_DIR)

# ---- scratch run-config (override only the data path; everything else stays default)
RUN_DIR = REPO / "bdb27_run"
RUN_DIR.mkdir(parents=True, exist_ok=True)
RUN_CFG_PATH = RUN_DIR / "config.yaml"
run_cfg = json.loads(json.dumps(CFG_REPO))          # deep copy (config is JSON-safe)
run_cfg.pop("_config_path", None)
run_cfg["paths"]["raw"] = str(DATA)                 # absolute -> resolve() passes it through
RUN_CFG_PATH.write_text(yaml.safe_dump(run_cfg, sort_keys=False), encoding="utf-8")
RUN_CFG = common.load_config(str(RUN_CFG_PATH))

OUTPUTS  = common.resolve(RUN_CFG, "outputs")
FIGDIR   = common.resolve(RUN_CFG, "figures")
FIGDIR.mkdir(parents=True, exist_ok=True)

# ---- which raw files are present? -----------------------------------------
RAW_FILES = ["combine_tracking.csv", "combine_results.csv", "players.csv",
             "games.csv", "player_play.csv",
             "game_tracking_2023.csv", "game_tracking_2024.csv", "game_tracking_2025.csv",
             "game_tracking_2023_sample.csv"]
present = {f: (DATA / f).exists() for f in RAW_FILES}
full_game_ok = all(present[f] for f in ("game_tracking_2023.csv", "game_tracking_2024.csv",
                                        "game_tracking_2025.csv"))
combine_ok   = present["combine_tracking.csv"] and present["combine_results.csv"]

if CLEAN:
    for d in ("audit", "envelope", "game", "models", "validate", "security", "security_models"):
        shutil.rmtree(OUTPUTS / d, ignore_errors=True)
    for f in ("thresholds.json", "run_log.csv"):
        try: (OUTPUTS / f).unlink()
        except FileNotFoundError: pass
    for f in FIGDIR.glob("*.png"):
        f.unlink()

# ---- shared output readers (used by the run + figure cells) -----------------
def out_path(*parts):
    return OUTPUTS.joinpath(*parts)

def read_csv(path, **kw):
    path = Path(path)
    return pd.read_csv(path, **kw) if path.exists() else pd.DataFrame()

def prov_of(df, fallback="UNVALIDATED SAMPLE OUTPUT"):
    if len(df) and "provenance" in df.columns:
        return str(df["provenance"].iloc[0])
    return fallback

def to_num(s):
    return pd.to_numeric(s, errors="coerce")

def is_true(s):
    return s.astype(str).str.strip().str.lower().isin(["true", "1", "yes"])

print(f"repo root      : {REPO}")
print(f"data dir       : {DATA}")
print(f"mode           : {MODE}   (force={FORCE}, clean={CLEAN})")
print(f"Combine files  : {'present' if combine_ok else 'MISSING'}")
print(f"full game files: {'present' if full_game_ok else 'ABSENT -> game-side stages are PENDING FULL RUN'}")
print(f"run config     : {RUN_CFG_PATH.relative_to(REPO)}")
print(f"outputs        : {OUTPUTS.relative_to(REPO)}/   figures -> {FIGDIR.relative_to(REPO)}/")

## 3. Run the pipeline in order

Each stage is invoked through its own entry point (so its `--mode` / `--force` / checkpoint
behaviour and the pre-registration **freeze gate** all apply exactly as on the command line).
A stage that needs data this machine does not have is caught and reported as
`PENDING FULL RUN` rather than crashing the notebook.

`06_security` is Combine-side (board + envelope), so it runs for real here; `07_security_models`
is game-side and emits a labelled `PENDING FULL RUN` placeholder in sample mode.

In [ ]:
STAGE_ORDER = ["01_audit", "02_combine_envelope", "06_security", "00_thresholds",
               "03_game_demand", "07_security_models", "00b_external_proxies",
               "04_models", "05_validate"]
GAME_SIDE  = {"01_audit", "00_thresholds", "03_game_demand", "07_security_models",
              "00b_external_proxies", "04_models", "05_validate"}

def run_stage(module_name, extra_args):
    """Run one src/ stage in-process via its own main(). Returns a status dict."""
    mod = importlib.import_module(module_name)          # numeric names are fine via importlib
    old = sys.argv
    sys.argv = [f"{module_name}.py", *[str(a) for a in extra_args]]
    buf = io.StringIO()
    t0, rc, err = time.time(), 0, ""
    try:
        with redirect_stdout(buf), redirect_stderr(buf):
            mod.main()
    except SystemExit as e:
        rc = 0 if e.code in (None, 0) else int(e.code)
    except BaseException as e:
        rc, err = 1, f"{type(e).__name__}: {e}"
        traceback.print_exc(file=buf)
    finally:
        sys.argv = old
    tail = [ln for ln in buf.getvalue().strip().splitlines() if ln.strip()][-6:]
    return {"stage": module_name, "ok": rc == 0, "rc": rc,
            "seconds": round(time.time() - t0, 2), "error": err, "tail": tail}

SUMMARY_FILE = {"03_game_demand": ("game", "game_demand_summary.csv"),
                "07_security_models": ("security_models", "security_models_summary.csv"),
                "04_models":      ("models", "models_summary.csv"),
                "05_validate":    ("validate", "validation_summary.csv")}

def stage_output_status(name):
    """Honest per-stage status from the stage's OWN outputs (PENDING vs OK vs UNVALIDATED)."""
    try:
        if name == "00_thresholds":
            p = out_path("thresholds.json")
            prov = json.loads(p.read_text())["provenance"] if p.exists() else None
            return "OK" if prov == "FULL" else f"PENDING FULL RUN (thresholds={prov})"
        if name == "01_audit":
            return "OK" if MODE == "full" else "PARTIAL (game-side c/h/i PENDING FULL RUN)"
        if name == "02_combine_envelope":
            p = out_path("envelope", "_DONE")
            m = json.loads(p.read_text()).get("mode") if p.exists() else None
            return "OK (FULL)" if m == "full" else "OK (sample)"
        if name == "06_security":
            p = out_path("security", "SUMMARY.json")
            if not p.exists():
                return "PENDING FULL RUN (security outputs absent)"
            s = json.loads(p.read_text())
            return f"OK ({s.get('mode')}; Analysis A appendix-only, D38)"
        if name == "00b_external_proxies":
            p = out_path(CFG_REPO["external"]["proxy_file"])
            if not p.exists():
                return "PENDING FULL RUN (external proxy absent)"
            return "OK" if MODE == "full" else "OK (sample)"
        if name == "07_security_models":
            # full/synthetic runs write SUMMARY.json; only the sample placeholder writes the CSV
            p = out_path("security_models", "SUMMARY.json")
            if p.exists():
                prov = str(json.loads(p.read_text()).get("provenance", ""))
                return "OK" if prov == "FULL" else "OK (synthetic)"
            p = out_path("security_models", "security_models_summary.csv")
            if p.exists():
                st = str(pd.read_csv(p)["status"].iloc[0])
                return st if st == "PENDING FULL RUN" else "OK"
            return "PENDING FULL RUN"
        sub, fn = SUMMARY_FILE[name]
        p = out_path(sub, fn)
        if p.exists():
            st = str(pd.read_csv(p)["status"].iloc[0])
            return st if st == "PENDING FULL RUN" else "OK"
    except Exception:
        pass
    return "OK"

FORCE_ARGS = ["--force"] if FORCE else []
RUN_RESULTS = []
print(f"running pipeline in '{MODE}' mode ...\n")
for name in STAGE_ORDER:
    res = run_stage(name, ["--mode", MODE, *FORCE_ARGS])
    if res["ok"]:
        res["status"] = stage_output_status(name)
    else:
        res["status"] = "PENDING FULL RUN" if name in GAME_SIDE else "FAILED"
    RUN_RESULTS.append(res)
    print(f"[{res['status']:<44}] {name:<20} {res['seconds']:>7.2f}s")
    if res["error"]:
        print(f"           {res['error']}")

## 4. Synthetic-fixture demonstration (game-side path)

The Pi does not hold the full game-tracking files, so the game-side stages emit
`PENDING FULL RUN` placeholders above. To show that path **actually runs**, this cell drives
the committed, schema-compliant fixture (`fixtures/synthetic/`) with a **planted binding
ceiling** and a **null** control. Fixture outputs are labelled *not a finding* — they prove
the plumbing, nothing about the NFL.

In [ ]:
SYNTH_RESULTS = []
if RUN_SYNTHETIC_DEMO:
    for regime in SYNTHETIC_REGIMES:
        print(f"--- synthetic regime: {regime} ---")
        for name in ("00_thresholds", "03_game_demand", "00b_external_proxies", "04_models", "05_validate"):
            res = run_stage(name, ["--synthetic", "--synthetic-regime", regime, *FORCE_ARGS])
            res["status"] = "OK (synthetic)" if res["ok"] else "FAILED"
            SYNTH_RESULTS.append(res)
            print(f"  [{res['status']:<14}] {name:<20} {res['seconds']:>7.2f}s"
                  + ("" if res["ok"] else f"  {res['error']}"))
    # security: the pre-Combine exposure/Analysis-A stage + the B–G incentive-exposure stage
    for regime in ("planted", "null"):
        res = run_stage("06_security", ["--synthetic", "--synthetic-regime", regime, *FORCE_ARGS])
        res["status"] = "OK (synthetic)" if res["ok"] else "FAILED"
        SYNTH_RESULTS.append(res)
        print(f"  [{res['status']:<14}] {'06_security':<20} {res['seconds']:>7.2f}s"
              + ("" if res["ok"] else f"  {res['error']}"))
    for regime in SECURITY_REGIMES:
        res = run_stage("07_security_models", ["--synthetic", "--synthetic-regime", regime, *FORCE_ARGS])
        res["status"] = "OK (synthetic)" if res["ok"] else "FAILED"
        SYNTH_RESULTS.append(res)
        print(f"  [{res['status']:<14}] {'07_security_models':<20} {res['seconds']:>7.2f}s"
              + ("" if res["ok"] else f"  {res['error']}"))
else:
    print("synthetic demo skipped (RUN_SYNTHETIC_DEMO=False)")

## 5. Security / incentive exposure (D38)

Reads `outputs/security_models/` (real if present, else the synthetic `planted` / `null` /
`selection` fixture). Everything here is **descriptive**: the headline is the single pre-declared
WR `moved_down` coefficient. A **positive** `group_coef` = `moved_down` has a **smaller**
Combine→game gap, i.e. the incentive hypothesis; a **negative** one is evidence **against** it.

**Group assignment is pre-Combine only** (`edp_trend > ε` ⇒ `moved_down`, else within-position
tertiles of `log(EDP_pre)`; the middle tertile is `unclassified`). Post-Combine EDP is never used.

In [ ]:
def security_dir():
    """Real security-model outputs if present, else the best synthetic fixture."""
    real = out_path("security_models", "b_primary.csv")
    if real.exists():
        return out_path("security_models"), False
    for reg in ("planted", "selection", "null"):
        d = out_path("security_models", "synthetic", reg)
        if (d / "b_primary.csv").exists():
            return d, True
    return None, None

SEC_DIR, SEC_SYNTH = security_dir()
if SEC_DIR is None:
    print("no security-model outputs yet — run 07_security_models (or the synthetic demo)")
else:
    sec_b = read_csv(SEC_DIR / "b_primary.csv")
    sec_c = read_csv(SEC_DIR / "c_end.csv")
    sec_x = read_csv(SEC_DIR / "c_exceedance.csv")
    sec_assign = read_csv(SEC_DIR / "group_assignment.csv")
    sec_sum = json.loads((SEC_DIR / "SUMMARY.json").read_text()) if (SEC_DIR / "SUMMARY.json").exists() else {}
    sec_prov = prov_of(sec_b)
    print(f"source: {SEC_DIR.relative_to(REPO)}   provenance: {sec_prov}")
    print("Analysis A: retired as a headline gate by D38 — appendix only.")
    print("Sign: a POSITIVE group_coef = moved_down has a SMALLER Combine→game gap (the hypothesis).")

    if len(sec_assign):
        st = sec_assign.groupby(["stratum", "group"]).size().reset_index(name="n")
        print("\nGroup counts (pre-Combine sort key):")
        display(st)
    if len(sec_b):
        b = sec_b[sec_b["spec"] == "primary_env_vmax"][
            ["scope", "n_obs", "n_players", "group_coef", "group_se", "group_p",
             "group_ci_lo", "group_ci_hi", "group_coef_corrected"]].copy()
        for c in ("group_coef", "group_se", "group_p", "group_ci_lo", "group_ci_hi",
                  "group_coef_corrected"):
            b[c] = to_num(b[c])
        print("\nB — primary effort-level group coefficient (player-clustered SEs):")
        display(b)
    if len(sec_x):
        print("\nC — exceedance (overall first, then by group), WR:")
        display(sec_x[sec_x["scope"] == "WR"][
            ["group", "n_efforts", "n_exceed", "exceedance_rate", "wilson_lo", "wilson_hi"]])
    if len(sec_c):
        c = sec_c[(sec_c["scope"] == "WR") & (sec_c["basis"] == "mean_ceiling")][
            ["percentile", "log_edp_pre", "fitted_gap", "se", "ci_lo", "ci_hi"]].copy()
        for col in ("percentile", "log_edp_pre", "fitted_gap", "se", "ci_lo", "ci_hi"):
            c[col] = to_num(c[col])
        print("\nC — fitted gap by log(EDP_pre) percentile (WR, mean ceiling):")
        display(c)
    if SEC_SYNTH:
        print("\nSYNTHETIC FIXTURE (not a finding) — these numbers prove the plumbing only.")

## 6. Writeup figures (≤5)

Figures are rendered from the pipeline outputs into `outputs/figures/` and shown inline.
Pre-full-run, **Figure 1** is real (the Combine file is complete) and the game-side figures fall
back to the synthetic fixture, carrying its *not a finding* label. When `MODE="full"` has produced
real outputs, the same code picks those up automatically and labels them `FULL`.

**Figure slide (D38):** the old *feature-selection ICC/VIF* bar chart is **retired** — its data
remain in `envelope/feature_selection.csv` and are reported as a Methods table. The slot now carries
the **security / incentive-exposure** figure, so the committed set stays at **5**.

### Visualisation briefs

| # | chart_type | data_source | columns_required | output_filename |
|---|---|---|---|---|
| 1 | line + band | `envelope/per_player_envelope.csv` | `combine_position, env_vmax, env_tau, peak_speed` | `fig1_combine_envelope_decay.png` |
| 2 | 3-panel (counts / coef / gap) | `security_models/{group_assignment,b_primary,c_end}.csv` | `stratum, group, scope, group_coef, group_ci_lo, group_ci_hi, percentile, fitted_gap` | `fig2_security_incentive_exposure.png` |
| 3 | grouped bar | `models/*/exceedance_utilization.csv` | `pooling, exceedance_rate, u_median` | `fig3_exceedance_utilization.png` |
| 4 | line + error bars | `models/*/model_coefficients.csv` | `pooling, tau, estimate, ci_lo, ci_hi, unstable` | `fig4_quantile_slope_vs_tau.png` |
| 5 | multi-panel | `validate/*/{cv_results,measurement_error,permutation}.csv` | `tau, model, pinball, slope_uncorrected, slope_corrected, p_two_sided` | `fig5_validation_cv_permutation.png` |

*Retired:* `fig2_feature_selection_icc_vif.png` (now a Methods table, not a figure).

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

# ---- figure helpers (output readers are defined in the bootstrap cell) ------
def save_and_show(fig, filename):
    path = FIGDIR / filename
    fig.savefig(path, dpi=FIG_DPI, bbox_inches="tight")
    if SHOW_FIGURES:
        plt.show()
    print(f"saved -> {path.relative_to(REPO)}   ({path.stat().st_size} bytes)")
    return path

def models_series():
    """(label, dir, is_synthetic) tuples for Phase-4 outputs: real if present, else synthetic."""
    real = out_path("models", "model_coefficients.csv")
    if real.exists():
        return [("FULL", out_path("models"), False)]
    out = []
    for reg in ("binding", "null"):
        d = out_path("models", "synthetic", reg)
        if (d / "model_coefficients.csv").exists():
            out.append((f"synthetic:{reg}", d, True))
    return out

def validate_series():
    real = out_path("validate", "cv_results.csv")
    if real.exists():
        return [("FULL", out_path("validate"), False)]
    out = []
    for reg in ("binding", "null"):
        d = out_path("validate", "synthetic", reg)
        if (d / "cv_results.csv").exists():
            out.append((f"synthetic:{reg}", d, True))
    return out

FIGURES = []

### Figure 1 — Combine ceiling envelope (`v(t)=env_vmax·(1−e^{−t/env_tau})`)

In [ ]:
env = read_csv(out_path("envelope", "per_player_envelope.csv"))
if not len(env):
    print("envelope outputs missing — run 02_combine_envelope first")
else:
    prov1 = prov_of(env)
    t = np.linspace(0.0, 3.0, 200)
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 4.2))
    for pos, col in (("WR", "C0"), ("TE", "C1")):
        sub = env[env["combine_position"] == pos]
        vm, tau = to_num(sub["env_vmax"]).to_numpy(), to_num(sub["env_tau"]).to_numpy()
        m = np.isfinite(vm) & np.isfinite(tau) & (tau > 0)
        if not m.any():
            continue
        curves = vm[m, None] * (1.0 - np.exp(-t[None, :] / tau[m, None]))
        a1.plot(t, np.median(curves, 0), color=col, label=f"{pos} median (n={m.sum()})")
        a1.fill_between(t, np.percentile(curves, 25, 0), np.percentile(curves, 75, 0),
                        color=col, alpha=0.18)
        a2.scatter(to_num(sub["peak_speed"]), to_num(sub["env_vmax"]),
                   s=14, alpha=0.6, color=col, label=pos)
    a1.set_xlabel("time from movement onset (s)"); a1.set_ylabel("speed (yd/s)")
    a1.set_title("Fitted ceiling envelope v(t)"); a1.legend(fontsize=8)
    lim = list(a2.get_xlim()) + list(a2.get_ylim())
    lo, hi = min(lim), max(lim)
    a2.plot([lo, hi], [lo, hi], "k--", lw=1, label="y=x")
    a2.set_xlabel("model-free peak speed (yd/s)"); a2.set_ylabel("fitted ceiling env_vmax (yd/s)")
    a2.set_title("Fitted ceiling vs raw peak"); a2.legend(fontsize=8)
    fig.suptitle(f"Fig 1 — Combine ceiling envelope   [{prov1}]", fontsize=10)
    FIGURES.append(save_and_show(fig, "fig1_combine_envelope_decay.png"))

### Figure 2 — Security / incentive exposure (group counts, B contrast, C dose-response)

In [ ]:
SEC_DIR, SEC_SYNTH = security_dir()
if SEC_DIR is None:
    print("no security-model outputs yet (run 07_security_models, or the synthetic demo)")
else:
    assign = read_csv(SEC_DIR / "group_assignment.csv")
    b = read_csv(SEC_DIR / "b_primary.csv")
    c = read_csv(SEC_DIR / "c_end.csv")
    prov2 = prov_of(b)

    fig, (a1, a2, a3) = plt.subplots(1, 3, figsize=(14, 4.3))

    # (1) group counts by stratum
    if len(assign):
        g = assign.groupby(["stratum", "group"]).size().unstack(fill_value=0)
        idx = np.arange(len(g.index)); w = 0.8 / max(len(g.columns), 1)
        for j, col in enumerate(g.columns):
            a1.bar(idx + (j - (len(g.columns) - 1) / 2) * w, g[col], width=w, label=col)
        a1.set_xticks(idx); a1.set_xticklabels(list(g.index), rotation=15)
        a1.set_ylabel("players"); a1.legend(fontsize=8)
    a1.set_title("pre-Combine sort key")

    # (2) B group coefficient by scope (primary spec)
    if len(b):
        bb = b[b["spec"] == "primary_env_vmax"].copy()
        bb["_c"] = to_num(bb["group_coef"]); bb["_lo"] = to_num(bb["group_ci_lo"])
        bb["_hi"] = to_num(bb["group_ci_hi"])
        xs = np.arange(len(bb))
        a2.errorbar(xs, bb["_c"], yerr=[bb["_c"] - bb["_lo"], bb["_hi"] - bb["_c"]],
                    fmt="o", capsize=4, color="C0")
        a2.axhline(0, ls="--", c="k", lw=1)
        a2.set_xticks(xs); a2.set_xticklabels(bb["scope"])
        a2.set_ylabel("group coefficient (log speed)")
    a2.set_title("B: moved_down vs high_secure")

    # (3) C fitted gap by log(EDP) percentile
    if len(c):
        cc = c[(c["scope"] == "WR") & (c["basis"] == "mean_ceiling")].sort_values("percentile")
        if len(cc):
            x = to_num(cc["percentile"]); yv = to_num(cc["fitted_gap"])
            lo, hi = to_num(cc["ci_lo"]), to_num(cc["ci_hi"])
            a3.errorbar(x, yv, yerr=[yv - lo, hi - yv], fmt="-o", capsize=4)
        a3.set_xlabel("log(EDP_pre) percentile"); a3.set_ylabel("fitted gap (log speed)")
    a3.set_title("C: which end drives it (WR)")

    fig.suptitle(f"Fig 2 — Security / incentive exposure   [{prov2}]")
    fig.tight_layout()
    FIGURES.append(save_and_show(fig, "fig2_security_incentive_exposure.png"))
    print("A positive group coefficient = moved_down has the SMALLER gap (the hypothesis).")
    if SEC_SYNTH:
        print("SYNTHETIC FIXTURE (not a finding).")

### Figure 3 — Exceedance & utilization (game side)

In [ ]:
series = models_series()
if not series:
    print("no Phase-4 model outputs yet (run 04_models, or the synthetic demo)")
else:
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 4.2))
    pools = ["pooled", "WR", "TE"]
    x = np.arange(len(pools)); w = 0.8 / max(len(series), 1)
    synth = False
    for i, (label, d, is_synth) in enumerate(series):
        synth = synth or is_synth
        ex = read_csv(Path(d) / "exceedance_utilization.csv")
        if not len(ex):
            continue
        ex = ex.set_index("pooling")
        xs = x + (i - (len(series) - 1) / 2) * w
        a1.bar(xs, [to_num(ex["exceedance_rate"]).get(p, np.nan) for p in pools],
               width=w, label=label)
        a2.bar(xs, [to_num(ex["u_median"]).get(p, np.nan) for p in pools], width=w, label=label)
    a1.axhline(0.25, ls="--", c="r", lw=1); a1.text(0.01, 0.26, "reframe >0.25",
                                                    color="r", fontsize=7, transform=a1.get_yaxis_transform())
    a1.set_xticks(x); a1.set_xticklabels(pools); a1.set_ylabel("exceedance rate")
    a1.set_title("Share of efforts beating the ceiling"); a1.legend(fontsize=8)
    a2.axhline(1.0, ls="--", c="k", lw=1); a2.set_xticks(x); a2.set_xticklabels(pools)
    a2.set_ylabel("median utilization u = effort / ceiling")
    a2.set_title("Utilization (1.0 = at ceiling)"); a2.legend(fontsize=8)
    tag = "SYNTHETIC FIXTURE (not a finding)" if synth else "FULL"
    fig.suptitle(f"Fig 3 — Exceedance & utilization   [{tag}]", fontsize=10)
    FIGURES.append(save_and_show(fig, "fig3_exceedance_utilization.png"))

### Figure 4 — Quantile slope vs τ (does the link strengthen toward the tail?)

In [ ]:
series = models_series()
if not series:
    print("no Phase-4 model outputs yet (run 04_models, or the synthetic demo)")
else:
    fig, ax = plt.subplots(figsize=(8.5, 4.6))
    styles = {"WR": "-o", "TE": "--s"}
    synth = False
    for label, d, is_synth in series:
        synth = synth or is_synth
        mc = read_csv(Path(d) / "model_coefficients.csv")
        if not len(mc):
            continue
        mc = mc[(mc["feature"] == "env_vmax") & (mc["controls"] == "without_draft")]
        for pool, ls in styles.items():
            sub = mc[mc["pooling"] == pool].sort_values("tau")
            if not len(sub):
                continue
            est = to_num(sub["reported"])          # blank (null) reported -> NaN -> not plotted
            lo, hi = to_num(sub["ci_lo"]), to_num(sub["ci_hi"])
            ax.errorbar(to_num(sub["tau"]), est, yerr=[est - lo, hi - est],
                        fmt=ls, capsize=3, alpha=0.9, label=f"{label} · {pool}")
            un = is_true(sub["unstable"]) & est.notna()
            ax.scatter(to_num(sub["tau"])[un], est[un], marker="x", s=45, color="k", zorder=5)
    ax.axhline(0.0, lw=1, color="k", alpha=0.5)
    ax.set_xlabel("τ (quantile)"); ax.set_ylabel("slope on ceiling env_vmax  (in-game speed per yd/s)")
    ax.set_title("Quantile slope vs τ  (×  = bootstrap-unstable; blanks = reported null)")
    ax.legend(fontsize=8)
    tag = "SYNTHETIC FIXTURE (not a finding)" if synth else "FULL"
    fig.suptitle(f"Fig 4 — Ceiling → game link across the tail   [{tag}]", fontsize=10)
    FIGURES.append(save_and_show(fig, "fig4_quantile_slope_vs_tau.png"))

### Figure 5 — Validation: player-grouped CV vs controls-only, measurement error, permutation

In [ ]:
series = validate_series()
if not series:
    print("no Phase-5 validation outputs yet (run 05_validate, or the synthetic demo)")
else:
    label, d, is_synth = series[0]         # binding (or real) is the reference
    cv = read_csv(Path(d) / "cv_results.csv")
    me = read_csv(Path(d) / "measurement_error.csv")
    pm = read_csv(Path(d) / "permutation.csv")
    fig, (a1, a2, a3) = plt.subplots(1, 3, figsize=(14, 4.2))

    if len(cv):
        g = cv.groupby(["tau", "model"])["pinball"].apply(lambda s: to_num(s).mean()).unstack()
        taus = list(g.index)
        x = np.arange(len(taus)); w = 0.38
        if "full" in g:      a1.bar(x - w/2, g["full"],          width=w, label="full")
        if "controls_only" in g: a1.bar(x + w/2, g["controls_only"], width=w, label="controls-only baseline")
        a1.set_xticks(x); a1.set_xticklabels([str(t) for t in taus])
    a1.set_xlabel("τ"); a1.set_ylabel("mean pinball loss (lower = better)")
    a1.set_title("Player-grouped CV vs baseline"); a1.legend(fontsize=8)

    if len(me):
        mt = to_num(me["tau"])
        a2.plot(mt, to_num(me["slope_uncorrected"]), "-o", label="uncorrected")
        a2.plot(mt, to_num(me["slope_corrected"]), "--s", label="ICC-attenuation corrected")
        if len(me) and "reliability_icc" in me:
            a2.set_title(f"Measurement error\n(reliability ICC ≈ {to_num(me['reliability_icc']).mean():.3f})",
                         fontsize=10)
    a2.axhline(0.0, lw=1, color="k", alpha=0.5)
    a2.set_xlabel("τ"); a2.set_ylabel("slope"); a2.legend(fontsize=8)

    if len(pm):
        r = pm.iloc[0]
        obs = abs(float(to_num(pd.Series([r["observed_slope"]]).iloc[0])))
        p = float(to_num(pd.Series([r["p_two_sided"]]).iloc[0]))
        a3.bar(["observed\n|slope|"], [obs], color="C3", width=0.5)
        a3.axhline(0.0, lw=1, color="k", alpha=0.5)
        a3.set_title(f"Permutation null (τ={r['tau']})\np = {p:.3f}  "
                     f"({int(r['n_ge'])}/{int(r['n_reps'])} perms ≥ obs)", fontsize=9)
        a3.set_ylabel("|slope|")
    tag = "SYNTHETIC FIXTURE (not a finding)" if is_synth else "FULL"
    fig.suptitle(f"Fig 5 — Validation   [{tag}]", fontsize=10)
    FIGURES.append(save_and_show(fig, "fig5_validation_cv_permutation.png"))

## 7. Run summary

A consolidated status for the whole run, the per-stage wall-time / peak-RAM table, and the
`PENDING FULL RUN` items. In `sample` mode every number is stamped `UNVALIDATED SAMPLE OUTPUT`;
`full` mode is the real analysis; synthetic fixtures are `SYNTHETIC FIXTURE (not a finding)`.

In [ ]:
def stage_status(res):
    return res.get("status") or ("OK" if res["ok"] else "FAILED")

summary = pd.DataFrame([{"stage": r["stage"], "status": stage_status(r),
                         "seconds": r["seconds"], "error": r["error"]} for r in RUN_RESULTS])
if SYNTH_RESULTS:
    summary = pd.concat([summary, pd.DataFrame(
        [{"stage": f"{r['stage']} [synthetic]", "status": stage_status(r),
          "seconds": r["seconds"], "error": r["error"]} for r in SYNTH_RESULTS])],
        ignore_index=True)

prov_file = out_path("thresholds.json")
prov = json.loads(prov_file.read_text())["provenance"] if prov_file.exists() else "n/a"
pending = summary.loc[summary["status"].str.contains("PENDING"), "stage"].tolist()

print(f"MODE = {MODE}   ·   thresholds provenance = {prov}")
print(f"label            : " + ("UNVALIDATED SAMPLE OUTPUT" if MODE != "full"
                               else "FULL (real analysis)"))
print(f"figures written  : {len(FIGURES)} of max 5  ->  {FIGDIR.relative_to(REPO)}/"
      "   (retired: fig2_feature_selection_icc_vif.png; replaced by fig2_security_incentive_exposure.png)")
print(f"PENDING FULL RUN : {pending if pending else 'none'}")
display(summary)

# ---- runtime / peak-RAM table (from outputs/run_log.csv) --------------------
rl = out_path("run_log.csv")
if rl.exists():
    log = read_csv(rl)
    log["_s"] = to_num(log["wall_s"]); log["_m"] = to_num(log["peak_ram_mb"])
    rt = (log[log["_s"] > 0].groupby("stage")
          .agg(runs=("_s", "size"), median_wall_s=("_s", "median"),
               max_wall_s=("_s", "max"), peak_ram_mb=("_m", "max"))
          .reset_index().sort_values("stage"))
    print("\nwall time / peak RAM per stage (outputs/run_log.csv):")
    display(rt)

### Next steps

* **On the Pi / laptop without the full tracking files:** leave `MODE="sample"`. Stages that
  need the full game data report `PENDING FULL RUN`; the synthetic demo shows the game-side
  path runs end-to-end.
* **With the full data (laptop or Kaggle):** put `game_tracking_{2023,24,25}.csv` (and the
  Combine files) in one folder, set `DATA_DIR` to it (or attach the Kaggle dataset and let
  auto-detect find it), set `MODE="full"`, then **Run All** again. The freeze gate
  (`notes/prereg.md`, frozen) plus the audit go/no-go gate are enforced by the engine before
  any full game-side stage runs.

Engine: `src/` · Spec: `TASK.md` · Pre-registration: `notes/prereg.md` · Decisions:
`notes/decisions.md`.